### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following.

1. Tracking agent behavior with logging, analytics and debugging
2. Transforming prompts, tool selection and output formatting.
3. Adding retries, fallbacks and early termination logic.
4. Applying rate limit, guardrails and PII detection.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY") 

### Summarization MiddleWare

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. SUmmarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [2]:
### Summarization MiddleWare
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Message Based summarization
agent = create_agent(
    model= "groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)


In [3]:
config = {"configurable": {"thread_id": "test-1"}}

In [4]:
# Alternative test data
question = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?"
]

for q in question:
    response = agent.invoke({"messages" : [HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response["messages"])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='6880287f-d81f-471b-bcd6-f85960c22fc0'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'The user asks a simple math question. Answer: 4.'}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 78, 'total_tokens': 111, 'completion_time': 0.068143977, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.003087148, 'prompt_tokens_details': None, 'queue_time': 0.343294911, 'total_time': 0.071231125}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4200b3f836', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a11fd0-743b-7522-93b6-bf631ab81750-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 33, 'total_tokens': 111, 'output_token_details': {'reasoning': 14}})]}
Messages: 2
M

### Based on Token Size


In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotel(city: str) -> str:
    """Search hotels - return long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Bugget Stay - 3 star, $75/night, free wifi
    """

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_hotel],
    checkpointer= InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("tokens", 550),
            keep=("tokens", 200)
        )
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token

In [6]:
# Run Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(    
        {"messages" : [HumanMessage(content="find hotels in {city}")]},
        config = config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~ {tokens} tokens, {len(response["messages"])} messages")
    print(f"{(response["messages"])}")


# video stoped  - 2:16:17

Paris: ~ 25 tokens, 2 messages
[HumanMessage(content='find hotels in {city}', additional_kwargs={}, response_metadata={}, id='7f875a8f-ca89-44ed-8bc3-fb9675c2983a'), AIMessage(content='Sure thing! Could you let me know which city you’d like to search for hotels in?', additional_kwargs={'reasoning_content': 'User says: "find hotels in {city}". They used a placeholder {city}. Probably they want to search hotels in a given city, but they didn\'t specify which city. We need to ask clarification.'}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 131, 'total_tokens': 201, 'completion_time': 0.145902481, 'completion_tokens_details': {'reasoning_tokens': 42}, 'prompt_time': 0.005776439, 'prompt_tokens_details': None, 'queue_time': 0.380782569, 'total_time': 0.15167892}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_ca5edfaab2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a11fd0-a

### Based On Fraction

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotel(city: str) -> str :
    """ Search Hotels. """
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# Low fraction for testing!
agent = create_agent(
    model = "groq:openai/gpt-oss-120b", # 
    tools = [search_hotel],
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("fraction", 0.005), # 0.5% = ~640 tokens
            keep=("fraction", 0.002) # 0.2% = ~256 tokens
        )
    ]
)

config = {"configurable" : {"thread_id": "test-1"}}

# Token Counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages" : [HumanMessage(content=f"Hotels in {city}")]},
        config = config
    )

    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000 # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction: .4%}), {len(response["messages"])} msg")
    print(response["messages"])

Paris: ~262 tokens ( 0.2047%), 4 msg
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='f3eb0366-82a8-4e67-b552-09bb71131b73'), AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants hotels in Paris. We should call search_hotel function with city "Paris".', 'tool_calls': [{'id': 'fc_c9c4b7db-83db-418c-a66b-d208b9572f67', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotel'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 120, 'total_tokens': 167, 'completion_time': 0.100185759, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.005116907, 'prompt_tokens_details': None, 'queue_time': 0.336245695, 'total_time': 0.105302666}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_1d982b31b2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a11fec-f3b8-7da0-b73e-f72cd